In [ ]:
import pandas as pd
import matplotlib as plt
import numpy as np
import cenpy

In [ ]:
def fetch_real_data():
    
    # Get 2023 data
    acs_2023 = cenpy.products.ACS(2023)
    data_2023 = acs_2023.from_state(
        state='North Carolina',
        level='county',
        variables=[
            'B08301_021E',  # WFH workers
            'B08301_001E',  # Total workers
            'B19013_001E',  # Median income
            'B28002_004E',  # Has broadband
            'B28002_001E',  # Total HH
        ]
    )
    
    # Get 2019 baseline
    acs_2019 = cenpy.products.ACS(2019)
    data_2019 = acs_2019.from_state(
        state='North Carolina',
        level='county',
        variables=['B08301_021E', 'B08301_001E']
    )
    
    # Merge
    df = pd.merge(data_2019, data_2023, on='GEOID', suffixes=('_2019', '_2023'))
    
    return df

In [ ]:
def calculate_metrics(df_counties, df_tracts):
    """Add derived columns"""
    
    # County metrics
    df_counties['WFH_Rate_2019'] = (df_counties['WFH_Workers_2019'] / df_counties['Total_Workers_2019'] * 100).round(1)
    df_counties['WFH_Rate_2023'] = (df_counties['WFH_Workers_2023'] / df_counties['Total_Workers_2023'] * 100).round(1)
    df_counties['WFH_Growth_Pct'] = ((df_counties['WFH_Rate_2023'] - df_counties['WFH_Rate_2019']) / df_counties['WFH_Rate_2019'] * 100).round(1)
    df_counties['WFH_Absolute_Growth'] = df_counties['WFH_Workers_2023'] - df_counties['WFH_Workers_2019']
    df_counties['Broadband_Gap'] = 100 - df_counties['Broadband_Pct']
    df_counties['WFH_Density'] = (df_counties['WFH_Workers_2023'] / df_counties['Land_Area_SqMi']).round(1)
    df_counties['Market_Value_Score'] = (df_counties['WFH_Workers_2023'] * df_counties['Median_Income'] * df_counties['Broadband_Gap'] / 1e9).round(2)
    
    # Tract metrics
    df_tracts['WFH_Rate'] = (df_tracts['WFH_Workers'] / df_tracts['Total_Workers'] * 100).round(1)
    df_tracts['Broadband_Gap'] = 100 - df_tracts['Broadband_Pct']
    df_tracts['Market_Value'] = (df_tracts['WFH_Workers'] * df_tracts['Median_Income'] * df_tracts['Broadband_Gap'] / 1e6).round(2)
    
    return df_counties, df_tracts


In [ ]:
def chart_1_wfh_growth(df_counties):
    """Bar chart: 2019 vs 2023 WFH rates by county"""
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(df_counties))
    width = 0.35
    
    ax.bar(x - width/2, df_counties['WFH_Rate_2019'], width, label='2019', alpha=0.8, color='steelblue')
    ax.bar(x + width/2, df_counties['WFH_Rate_2023'], width, label='2023', alpha=0.8, color='coral')
    
    ax.set_xlabel('County', fontsize=12, fontweight='bold')
    ax.set_ylabel('WFH Rate (%)', fontsize=12, fontweight='bold')
    ax.set_title('WFH Growth: Charlotte Metro (2019 vs 2023)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df_counties['County'], rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Add growth labels
    for i, (idx, row) in enumerate(df_counties.iterrows()):
        ax.text(i, row['WFH_Rate_2023'] + 0.5, f"+{row['WFH_Growth_Pct']}%", 
               ha='center', va='bottom', fontsize=9, fontweight='bold', color='darkred')
    
    plt.tight_layout()
    return fig


In [ ]:
df_counties = fetch_real_data()
df_counties, df_tracts = calculate_metrics(df_counties, df_tracts)

fig1 = chart_1_wfh_growth(df_counties)
plt.savefig('chart1_wfh_growth.png', dpi=150, bbox_inches='tight')
print("✓ Chart 1: WFH Growth")

NotImplementedError: The requested year (2023) is too early/late. Only 2017, 2018, or 2019 are supported.